# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-4o-mini"
print("client ready:", MODEL)

client ready: gpt-4o-mini


## 1) Task 1 — 단발 호출 + 토큰/비용 확인

In [2]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
        {"role": "user", "content": "사내 챗봇을 한 문장으로 소개해줘."},
    ],
    temperature=0.7,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

사내 챗봇은 직원들이 정보를 쉽게 얻고 업무를 효율적으로 수행할 수 있도록 지원하는 인공지능 기반의 대화형 도구입니다.
usage: CompletionUsage(completion_tokens=36, prompt_tokens=39, total_tokens=75, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


### 스트리밍 — 토큰이 생성되는 대로 출력

In [3]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "LLM 을 3줄로 설명해줘."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="")
print()

LLM(대형 언어 모델)은 대량의 텍스트 데이터를 학습하여 자연어를 이해하고 생성할 수 있는 인공지능 시스템입니다. 이를 통해 질문에 답변하거나, 대화를 나누거나, 글쓰기 등 다양한 언어 관련 작업을 수행할 수 있습니다. LLM은 GPT 같은 아키텍처를 기반으로 하며, 사용자의 입력에 따라 적절한 응답을 생성하는 능력을 가지고 있습니다.


## 2) Task 2 — 리뷰 100개 긍/부정 분류 (LLM)

실습 1의 산출물 `reviews_clean.parquet` 에서 100건을 샘플링해 분류한다.
- `temperature=0` (분류는 결정적이어야 함)
- **JSON 응답 강제** (`response_format`) — 파싱 안전

In [4]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]                      # 중립 제외 (실습 2 와 동일 기준)
df["label"] = (df["rating"] >= 4).astype(int)   # 정답: 4~5 긍정(1), 1~2 부정(0)
sample = df.sample(100, random_state=42).reset_index(drop=True)
print(sample["label"].value_counts())
sample[["clean_text", "label"]].head()

label
1    62
0    38
Name: count, dtype: int64


,clean_text,label
0,색감이 사진이랑 똑같아서 만족합니다,1
1,사진과 색이 완전히 달라서 실망했어요,0
2,광고 지금 구매하면 사은품 증정 배송이 정말 빨라서 깜짝 놀랐어요,1
3,고객센터 연결도 안 되고 최악이에요,0
4,가성비 최고입니다 또 살게요,1


In [5]:
import json

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    'JSON 으로만 답한다: {"label": 0 또는 1}'
)

def classify(text: str) -> tuple[int, int]:   # (라벨, 사용 토큰) 을 함께 반환
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},   # JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    return int(data["label"]), resp.usage.total_tokens   # 라벨 + 비용계산용 토큰

# 1건 점검 — (라벨, 토큰) 튜플이 찍힌다
label, used = classify(sample.loc[0, "clean_text"])
print("라벨:", label, "/ 토큰:", used)

라벨: 1 / 토큰: 81


In [6]:
preds, tokens = [], 0
for t in sample["clean_text"]:
    label, used = classify(t)
    preds.append(label)
    tokens += used
sample["pred"] = preds
print("총 토큰:", tokens)

총 토큰: 8554


### sklearn(실습 2) 과 비교 — 정확도

In [12]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["label"], sample["pred"])

print(f"LLM 정확도: {acc:.3f}")

print(classification_report(sample["label"], sample["pred"], digits=3))

# 실습 2 sklearn report 생성 (코드 일부 발췌, 간략하게) 
import time
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# 준비 데이터 (sample과 같은 기준 정리, 100개가 아닌 전체 데이터에서 진행)
df2 = pd.read_parquet("../data/reviews_clean.parquet")
df2 = df2.dropna(subset=["rating"]).copy()
df2 = df2[df2["rating"] != 3]
df2["label"] = (df2["rating"] >= 4).astype(int)
X2 = df2["clean_text"].fillna("")
y2 = df2["label"]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y2, test_size=0.2, stratify=y2, random_state=42)

# 비교 대상 모델 정의
N_FIT = 3000
X_fit2, y_fit2 = X_tr2.iloc[:N_FIT], y_tr2.iloc[:N_FIT]
candidates = {
    "LogReg (baseline)": Pipeline([
        ("vec", TfidfVectorizer(max_features=5000)),
        ("clf", LogisticRegression(max_iter=1000)),
    ]),
    "LogReg + ngram(1,2)": Pipeline([
        ("vec", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.9)),
        ("clf", LogisticRegression(max_iter=1000)),
    ]),
    "RandomForest": Pipeline([
        ("vec", TfidfVectorizer(max_features=5000)),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42)),
    ]),
    "GradientBoosting": Pipeline([
        ("vec", TfidfVectorizer(max_features=5000)),
        ("clf", GradientBoostingClassifier(random_state=42)),
    ]),
}

rows = []
for name, model in candidates.items():
    t0 = time.perf_counter()
    model.fit(X_fit2, y_fit2)
    train_sec = time.perf_counter() - t0
    pred = model.predict(X_te2)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_te2, pred),
        "f1": f1_score(y_te2, pred),
        "train_sec": train_sec,
    })

report = pd.DataFrame(rows).set_index("model").round(3)

# LLM과 sklearn 비교표 생성
sklearn_report = report.copy()

llm_acc = acc
llm_cost = cost_100 * 100
llm_speed = "느림 (API 호출)"
llm_consistency = "높음 (deterministic)"

compare_df = sklearn_report.copy()
compare_df["cost($) for 10k"] = float('nan')
compare_df["speed"] = "빠름"
compare_df["consistency"] = "중간"

compare_df.loc["LogReg (baseline)", "cost($) for 10k"] = 0.0
compare_df.loc["LogReg (baseline)", "speed"] = "매우 빠름"
compare_df.loc["LogReg (baseline)", "consistency"] = "높음"

compare_df.loc["LLM GPT-4o-mini"] = [llm_acc, None, None, llm_cost, llm_speed, llm_consistency]

compare_df = compare_df.reset_index().rename(columns={"index": "model"})

import pandas as pd
from IPython.display import display

display(compare_df.round(3))


LLM 정확도: 1.000
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        38
           1      1.000     1.000     1.000        62

    accuracy                          1.000       100
   macro avg      1.000     1.000     1.000       100
weighted avg      1.000     1.000     1.000       100



,model,accuracy,f1,train_sec,cost($) for 10k,speed,consistency
0,LogReg (baseline),0.998,0.999,0.029,0.000,매우 빠름,높음
1,"LogReg + ngram(1,2)",0.998,0.999,0.055,NaN,빠름,중간
2,RandomForest,0.998,0.999,0.285,NaN,빠름,중간
3,GradientBoosting,0.992,0.993,0.207,NaN,빠름,중간
4,LLM GPT-4o-mini,1.000,None,None,0.128,느림 (API 호출),높음 (deterministic)


## 3) Task 3 — 비용 측정 + 1만 건 추산

In [8]:
# gpt-4o-mini 단가 (2026 기준, 변동 가능) — 슬라이드 '비용·운영 한눈에' 참고
# 입력 0.15/M, 출력 0.60/M $. 분류는 출력이 5토큰 내외로 매우 짧아 출력 비용은 무시하고
# total_tokens 를 입력 단가로 보수적으로 추정한다.
PRICE_IN = 0.15 / 1_000_000   # 입력 토큰당 $

avg_tokens = tokens / len(sample)
cost_100   = tokens * PRICE_IN                 # 보수적으로 입력 단가 적용
print(f"평균 토큰/건: {avg_tokens:.1f}")
print(f"100건 비용: ${cost_100:.4f}")
print(f"1만 건 추산: ${cost_100 * 100:.2f}")
# TODO: ML vs LLM — '언제 무엇을 쓸지' 한 줄 결론을 적는다

평균 토큰/건: 85.5
100건 비용: $0.0013
1만 건 추산: $0.13


## 회고 / 산출물
- [ ] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [ ] 한 줄 결론 — *대량·단순 분류는 ___, 유연·복잡 추론은 ___*

In [14]:

reviews = [
    "와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다",
    "품질 최고예요~ 환불하고 싶을 만큼",
    "빠른 배송 감사합니다 일주일이나 걸려서요",
]

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "리뷰가 긍정이면 1, 부정이면 0을 고른다. "
    "확신(confidence) 수치를 0~1 사이 실수로 표시한다. "
    'JSON 형식으로만 답해라: {"label": 0 또는 1, "confidence": 0.x}'
)

def classify_with_confidence(text: str):
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text}
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    return data["label"], data["confidence"]

# 테스트 - 3개 리뷰 분류
for r in reviews:
    label, conf = classify_with_confidence(r)
    print(f"리뷰: {r}\n긍정(1)/부정(0): {label}, 확신: {conf}\n")

리뷰: 와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다
긍정(1)/부정(0): 0, 확신: 0.85

리뷰: 품질 최고예요~ 환불하고 싶을 만큼
긍정(1)/부정(0): 1, 확신: 0.85

리뷰: 빠른 배송 감사합니다 일주일이나 걸려서요
긍정(1)/부정(0): 0, 확신: 0.8



In [16]:
import pandas as pd

import json

import re



# reviews.csv 로드

reviews_df = pd.read_csv("../data/reviews.csv")



# 리뷰 텍스트 컬럼 선택 (reviews.csv 구조: text/clean_text 등 자동 탐색)

text_col = "clean_text" if "clean_text" in reviews_df.columns else "text"



# 1) 결측치 제거

reviews_df = reviews_df.dropna(subset=[text_col]).copy()



# 2) 홍보/광고성 문구 제거 (간단 규칙 기반)

promo_pattern = re.compile(r"(광고|협찬|구매처|쿠폰|이벤트|체험단|http[s]?://|www\.)", re.IGNORECASE)

reviews_df = reviews_df[~reviews_df[text_col].str.contains(promo_pattern, na=False)]



# 샘플 20개만 테스트

texts = reviews_df[text_col].head(20).tolist()



SYSTEM = (

    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "

    "리뷰가 긍정이면 1, 부정이면 0을 고른다. "

    'JSON 형식으로만 답해라: {"label": 0 또는 1}'

)



def classify_review(text: str):

    resp = client.chat.completions.create(

        model=MODEL,

        temperature=0,

        response_format={"type": "json_object"},

        messages=[

            {"role": "system", "content": SYSTEM},

            {"role": "user", "content": text},

        ],

    )

    data = json.loads(resp.choices[0].message.content)

    return int(data["label"])



results = []

for t in texts:

    label = classify_review(t)

    results.append({"review": t, "pred": label})



pd.DataFrame(results).head(10)


/tmp/ipykernel_21175/2291175379.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  reviews_df = reviews_df[~reviews_df[text_col].str.contains(promo_pattern, na=False)]


,review,pred
0,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,0
1,가성비 최고입니다 또 살게요,1
2,사이즈도 딱 맞고 마감이 깔끔해요 🔥,1
3,재구매 의사 100% 강추합니다 😡,0
4,재구매 의사 100% 강추합니다,1
5,색감이 사진이랑 똑같아서 만족합니다 ㅠㅠ,1
6,사이즈도 딱 맞고 마감이 깔끔해요,1
7,재구매 의사 100% 강추합니다,1
8,무난합니다 그냥 평타 🔥,1
9,배송이 정말 빨라서 깜짝 놀랐어요,1
